In [0]:
# Silver — Dallas Cleaning (03_silver_dallas)
## Source : bronze.dallas_raw (latest batch)
## Target : silver.dallas_clean + silver.dallas_violations + silver.dallas_quarantine
## Rules  : null checks, zip format, zip range, score 0-100, ≥1 violation,
##          score≥90 → max 3 violations, dedup, violation parse + count

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from functools import reduce
from datetime import datetime

# Detect catalog dynamically
CATALOG = spark.sql("SELECT current_catalog()").first()[0]
if CATALOG in ("hive_metastore", "spark_catalog"):
    catalogs = [r[0] for r in spark.sql("SHOW CATALOGS").collect()]
    project_cats = [c for c in catalogs
                    if c not in ("hive_metastore", "spark_catalog",
                                 "system", "samples", "workspace")]
    CATALOG = project_cats[0] if project_cats else CATALOG

BRONZE_TABLE = f"{CATALOG}.bronze.dallas_raw"
SILVER_TABLE = f"{CATALOG}.silver.dallas_clean"
SILVER_VIOLS = f"{CATALOG}.silver.dallas_violations"
QUARANTINE   = f"{CATALOG}.silver.dallas_quarantine"

RUN_TS   = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
RUN_DATE = datetime.now().strftime("%Y-%m-%d")

# Dallas score range
DAL_SCORE_MIN = 0
DAL_SCORE_MAX = 100

print(f"Catalog       : {CATALOG}")
print(f"Bronze source : {BRONZE_TABLE}")
print(f"Silver target : {SILVER_TABLE}")
print(f"Run time      : {RUN_TS}")

In [0]:
# Verify bronze table exists
try:
    spark.table(BRONZE_TABLE).limit(1).collect()
    print(f"✓ Bronze table found: {BRONZE_TABLE}")
except Exception as e:
    raise Exception(f"Bronze table not found: {BRONZE_TABLE}\nError: {e}")

# Get latest batch ID
latest_batch = (
    spark.table(BRONZE_TABLE)
    .orderBy(F.col("_extract_ts").desc())
    .select("_batch_id")
    .first()[0]
)
print(f"Latest batch : {latest_batch}")

# Core columns to read
DAL_CORE = [
    "restaurant_name", "inspection_type", "inspection_date",
    "inspection_score", "street_address", "zip_code",
    "location_raw", "city_source", "_batch_id"
]

# All 25 violation sets
VIOL_COLS = (
    [f"viol_desc_{i}"   for i in range(1, 26)] +
    [f"viol_points_{i}" for i in range(1, 26)] +
    [f"viol_detail_{i}" for i in range(1, 26)] +
    [f"viol_memo_{i}"   for i in range(1, 26)]
)

df_dal = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("_batch_id") == latest_batch)
    .select(DAL_CORE + VIOL_COLS)
    # Dedup on restaurant_name + inspection_date + score
    .dropDuplicates(["restaurant_name", "inspection_date", "inspection_score"])
)

row_count = df_dal.count()
print(f"Rows after dedup : {row_count:,}")
print(f"Columns          : {len(df_dal.columns)}")
display(df_dal.select(
    "restaurant_name", "inspection_date",
    "inspection_score", "inspection_type", "zip_code"
).limit(3))

In [0]:
# Step 1: Cast types
df_dal_typed = (
    df_dal
    .withColumn("inspection_score",
        F.col("inspection_score").cast(DoubleType()))
    .withColumn("inspection_date",
        F.to_date(F.col("inspection_date"), "MM/dd/yyyy"))
)

# Step 2: Clean zip
# Dallas has formats: 75230-6114, 752124029 → extract first 5 digits
df_dal_typed = df_dal_typed.withColumn(
    "zip_code",
    F.regexp_extract(F.col("zip_code"), r"^(\d{5})", 1)
)
# Remove .0 if present, cast int then back to string
df_dal_typed = df_dal_typed.withColumn(
    "zip_code",
    F.col("zip_code").cast("int").cast("string")
)

# Step 3: Early filter — remove records with no violations at all
# Count non-null violation descriptions across all 25 slots
viol_count_expr = sum(
    F.when(
        F.col(f"viol_desc_{i}").isNotNull() &
        (F.col(f"viol_desc_{i}") != ""), 1
    ).otherwise(0)
    for i in range(1, 26)
)

df_dal_typed = df_dal_typed.withColumn(
    "raw_viol_count", viol_count_expr
)

before = df_dal_typed.count()

# Drop records with zero violations early
df_dal_typed = df_dal_typed.filter(F.col("raw_viol_count") >= 1)

# Drop records with score outside 0-100 early (includes negatives like -26)
df_dal_typed = df_dal_typed.filter(
    F.col("inspection_score").between(DAL_SCORE_MIN, DAL_SCORE_MAX)
)

after = df_dal_typed.count()

print(f"Before early filter : {before:,}")
print(f"After early filter  : {after:,}")
print(f"Dropped early       : {before - after:,}")

display(df_dal_typed.select(
    "restaurant_name", "inspection_date", "inspection_score",
    "inspection_type", "zip_code", "raw_viol_count"
).limit(5))

In [0]:
# Concat all 25 violation descriptions into one string for urgent/critical check
viol_concat_expr = F.concat_ws(" ",
    *[F.coalesce(F.col(f"viol_desc_{i}"), F.lit("")) for i in range(1, 26)]
)

df_dal_checked = (
    df_dal_typed

    # Add concatenated violations column first — used by last check
    .withColumn("all_viols_concat", viol_concat_expr)

    # Check 1: valid_business_name
    .withColumn("valid_business_name",
        F.col("restaurant_name").isNotNull() &
        (F.col("restaurant_name") != ""))

    # Check 2: valid_inspection_date
    .withColumn("valid_inspection_date",
        F.col("inspection_date").isNotNull())

    # Check 3: valid_inspection_type
    .withColumn("valid_inspection_type",
        F.col("inspection_type").isNotNull() &
        (F.col("inspection_type") != ""))

    # Check 4: valid_zip_code not null
    .withColumn("valid_zip_code",
        F.col("zip_code").isNotNull() &
        (F.col("zip_code") != "") &
        (F.col("zip_code") != "null"))

    # Check 5: valid_zip_format exactly ^[0-9]+$ as specified
    .withColumn("valid_zip_format",
        F.col("zip_code").rlike(r"^[0-9]+$"))

    # Check 6: violation score cannot be more than 100
    .withColumn("valid_score_range",
        F.col("inspection_score").between(DAL_SCORE_MIN, DAL_SCORE_MAX))

    # Check 7: score >= 90 must have max 3 violations
    .withColumn("valid_score_viol_rule",
        ~((F.col("inspection_score") >= 90) &
          (F.col("raw_viol_count") > 3)))

    # Check 8: inspection result cannot be PASS if any violation
    # contains Urgent or Critical terms
    # Dallas has no Results column — inspection_type is used instead
    # null violations_concat = auto True (no violations = no urgent terms)
    .withColumn("valid_no_urgent_pass",
        F.when(
            F.col("all_viols_concat").isNull() |
            (F.col("all_viols_concat") == ""),
            F.lit(True)
        ).otherwise(
            ~(
                F.upper(F.col("inspection_type")).isin(
                    "PASS", "ROUTINE", "FOLLOW-UP"
                ) &
                F.col("all_viols_concat").rlike(r"(?i)(urgent|critical)")
            )
        ))

    # Overall: ALL checks must pass
    .withColumn("_all_checks_pass",
        F.col("valid_business_name")    &
        F.col("valid_inspection_date")  &
        F.col("valid_inspection_type")  &
        F.col("valid_zip_code")         &
        F.col("valid_zip_format")       &
        F.col("valid_score_range")      &
        F.col("valid_score_viol_rule")  &
        F.col("valid_no_urgent_pass")
    )
)

CHECK_COLS = [
    "valid_business_name",
    "valid_inspection_date",
    "valid_inspection_type",
    "valid_zip_code",
    "valid_zip_format",
    "valid_score_range",
    "valid_score_viol_rule",
    "valid_no_urgent_pass",
    "_all_checks_pass"
]

# Split good vs bad
df_dal_good = (
    df_dal_checked
    .filter(F.col("_all_checks_pass") == True)
    .drop(*CHECK_COLS, "all_viols_concat")
    .withColumn("_silver_ts",   F.lit(RUN_TS))
    .withColumn("_silver_date", F.lit(RUN_DATE))
    .withColumn("_dqx_status",  F.lit("PASS"))
)

df_dal_bad = (
    df_dal_checked
    .filter(
        F.col("_all_checks_pass").isNull() |
        (F.col("_all_checks_pass") == False)
    )
    .withColumn("_silver_ts",   F.lit(RUN_TS))
    .withColumn("_silver_date", F.lit(RUN_DATE))
    .withColumn("_dqx_status",  F.lit("FAIL"))
)

# Verify zero rows lost
original   = df_dal_typed.count()
good_count = df_dal_good.count()
bad_count  = df_dal_bad.count()
lost       = original - (good_count + bad_count)

print(f"Original rows : {original:,}")
print(f"GOOD rows     : {good_count:,}")
print(f"BAD rows      : {bad_count:,}")
print(f"Rows lost     : {lost}  ← must be 0")

print("\nFailed check breakdown:")
display(
    df_dal_bad.select(*CHECK_COLS)
    .groupBy(*[c for c in CHECK_COLS if c != "_all_checks_pass"])
    .count()
    .orderBy(F.desc("count"))
    .limit(10)
)

In [0]:
# Unpivot wide format to long — one row per violation
dfs = []
for i in range(1, 26):
    dfs.append(
        df_dal_good.select(
            "restaurant_name",
            "inspection_date",
            "inspection_score",
            "inspection_type",
            "street_address",
            "zip_code",
            "location_raw",
            "city_source",
            "_batch_id",
            F.col(f"viol_desc_{i}").alias("violation_description"),
            F.col(f"viol_points_{i}").cast(DoubleType()).alias("violation_points"),
            F.col(f"viol_detail_{i}").alias("violation_detail"),
            F.col(f"viol_memo_{i}").alias("inspector_comment"),
            F.lit(i).alias("violation_slot")
        )
        .filter(
            F.col("violation_description").isNotNull() &
            (F.col("violation_description") != "")
        )
    )

df_dal_long = reduce(lambda a, b: a.union(b), dfs)

# Extract violation code — Dallas format is *01, *10, *31
df_dal_long = df_dal_long.withColumn(
    "violation_code",
    F.regexp_extract(F.col("violation_description"), r"^\*?(\d+)", 1)
)

# Deduplicate — one row per restaurant + date + violation code
df_dal_long = df_dal_long.dropDuplicates(
    ["restaurant_name", "inspection_date", "violation_code"]
)

# Count distinct violations per inspection
viol_counts = (
    df_dal_long
    .groupBy("restaurant_name", "inspection_date")
    .agg(F.countDistinct("violation_code").alias("violation_count"))
)

# Join count back using alias to avoid column collision
df_dal_good = (
    df_dal_good.alias("g")
    .join(
        viol_counts.alias("v"),
        on=["restaurant_name", "inspection_date"],
        how="left"
    )
    .withColumn("violation_count",
        F.coalesce(F.col("v.violation_count"), F.lit(0)))
    .select(
        *[F.col(f"g.{c}") for c in df_dal_good.columns],
        F.col("violation_count")
    )
)

print(f"Dallas good inspections : {df_dal_good.count():,}")
print(f"Dallas violation rows   : {df_dal_long.count():,}")

display(df_dal_long.select(
    "restaurant_name", "inspection_date", "violation_code",
    "violation_description", "violation_points"
).limit(5))

In [0]:
# Write 1: Clean inspections
# Drop the wide violation columns before writing — not needed in Silver
viol_wide_cols = (
    [f"viol_desc_{i}"   for i in range(1, 26)] +
    [f"viol_points_{i}" for i in range(1, 26)] +
    [f"viol_detail_{i}" for i in range(1, 26)] +
    [f"viol_memo_{i}"   for i in range(1, 26)] +
    ["raw_viol_count"]
)

# Only drop columns that actually exist in df_dal_good
cols_to_drop = [c for c in viol_wide_cols if c in df_dal_good.columns]

df_dal_final = df_dal_good.drop(*cols_to_drop)

(
    df_dal_final
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)
print(f"✓ {SILVER_TABLE}")
print(f"  Rows    : {df_dal_final.count():,}")
print(f"  Columns : {len(df_dal_final.columns)}")

# Write 2: Violations long format
(
    df_dal_long
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_VIOLS)
)
print(f"\n✓ {SILVER_VIOLS}")
print(f"  Rows    : {df_dal_long.count():,}")

# Write 3: Quarantine
cols_to_drop_bad = [c for c in viol_wide_cols if c in df_dal_bad.columns]
df_dal_bad_final = df_dal_bad.drop(*cols_to_drop_bad)

(
    df_dal_bad_final
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE)
)
print(f"\n✓ {QUARANTINE}")
print(f"  Rows    : {df_dal_bad_final.count():,}")
print(f"\n_silver_ts  : {RUN_TS}")
print(f"_silver_date: {RUN_DATE}")

In [0]:
df_verify = spark.sql(f"""
    SELECT
        city_source,
        COUNT(*)                             AS total_inspections,
        ROUND(AVG(inspection_score), 2)      AS avg_score,
        ROUND(AVG(violation_count), 2)       AS avg_violations,
        MIN(inspection_score)                AS min_score,
        MAX(inspection_score)                AS max_score,
        SUM(CASE WHEN inspection_type = 'Routine'
                 THEN 1 ELSE 0 END)          AS routine_count,
        SUM(CASE WHEN inspection_type = 'Follow-up'
                 THEN 1 ELSE 0 END)          AS followup_count,
        SUM(CASE WHEN inspection_type = 'Complaint'
                 THEN 1 ELSE 0 END)          AS complaint_count,
        MIN(_silver_ts)                      AS first_loaded,
        MAX(_silver_ts)                      AS last_loaded
    FROM {SILVER_TABLE}
    GROUP BY city_source
""")
display(df_verify)

In [0]:
df_quar = spark.sql(f"""
    SELECT
        SUM(CASE WHEN valid_business_name    = false THEN 1 ELSE 0 END) AS failed_business_name,
        SUM(CASE WHEN valid_inspection_date  = false THEN 1 ELSE 0 END) AS failed_date,
        SUM(CASE WHEN valid_inspection_type  = false THEN 1 ELSE 0 END) AS failed_type,
        SUM(CASE WHEN valid_zip_code         = false THEN 1 ELSE 0 END) AS failed_zip_null,
        SUM(CASE WHEN valid_zip_format       = false THEN 1 ELSE 0 END) AS failed_zip_format,
        SUM(CASE WHEN valid_score_range      = false THEN 1 ELSE 0 END) AS failed_score_range,
        SUM(CASE WHEN valid_score_viol_rule  = false THEN 1 ELSE 0 END) AS failed_score_viol_rule,
        SUM(CASE WHEN valid_no_urgent_pass   = false THEN 1 ELSE 0 END) AS failed_urgent_pass,
        COUNT(*)                                                          AS total_quarantined
    FROM {QUARANTINE}
""")
display(df_quar)

display(spark.sql(f"DESCRIBE HISTORY {SILVER_TABLE}"))